In [1]:
# Install & Restart, Run From Cell 2
# !pip install flask scikit-learn umap-learn joblib numpy pandas scipy -q

In [6]:
import os, json, shutil
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/CareerSegmentation' # Make sure the path same as your CareerSegmentation path in drive
APP_DIR      = os.path.join(BASE_DIR, 'App')
TMPL_DIR     = os.path.join(APP_DIR,  'templates')
ARTIFACT_DIR = os.path.join(BASE_DIR, 'Outputs', 'artifacts')
MODELS_DIR   = os.path.join(BASE_DIR, 'Outputs', 'models')

os.makedirs(APP_DIR,  exist_ok=True)
os.makedirs(TMPL_DIR, exist_ok=True)

# Verify required artifacts
required = [
    os.path.join(ARTIFACT_DIR, 'mlb_encoders.pkl'),
    os.path.join(ARTIFACT_DIR, 'kmeans_final.pkl'),
    os.path.join(ARTIFACT_DIR, 'feature_names.json'),
    os.path.join(ARTIFACT_DIR, 'persona_labels.json'),
    os.path.join(ARTIFACT_DIR, 'model_config.json'),
    os.path.join(MODELS_DIR,   'cluster_profiles.json'),
    os.path.join(ARTIFACT_DIR,   'umap_model.pkl'),
]
print("Artifact check:")
all_ok = True
for path in required:
    exists = os.path.exists(path)
    print(f"  {'[V]' if exists else '[X]'} {os.path.basename(path)}")
    if not exists:
        all_ok = False

print(f"\n{'[Success] All artifacts present' if all_ok else '[Error] Missing artifactst'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifact check:
  [V] mlb_encoders.pkl
  [V] kmeans_final.pkl
  [V] feature_names.json
  [V] persona_labels.json
  [V] model_config.json
  [V] cluster_profiles.json
  [V] umap_model.pkl

[Success] All artifacts present


In [3]:
# Import Artifacts & App Folder

import shutil

LINK_ARTIFACT = os.path.join(APP_DIR, 'artifacts')
LINK_MODELS   = os.path.join(APP_DIR, 'models')

if os.path.exists(LINK_ARTIFACT):
    shutil.rmtree(LINK_ARTIFACT)
shutil.copytree(ARTIFACT_DIR, LINK_ARTIFACT)
print(f"artifacts/ copied ({len(os.listdir(LINK_ARTIFACT))} files)")

if os.path.exists(LINK_MODELS):
    shutil.rmtree(LINK_MODELS)
shutil.copytree(MODELS_DIR, LINK_MODELS)
print(f"models/ copied ({len(os.listdir(LINK_MODELS))} files)")

artifacts/ copied (8 files)
models/ copied (4 files)


In [4]:
# Cloudflare Tunnel

import subprocess, threading, time, os

os.chdir(APP_DIR)

# 1. Start Flask
def run_flask():
    subprocess.run(['python', 'app.py'])

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)
print("[Success] Flask running on port 5000")

# 2. Download cloudflared (binary Cloudflare Tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 3. Open Tunnel
import subprocess, re

proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait url
url = None
for _ in range(30):
    line = proc.stderr.readline()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break
    time.sleep(0.5)

if url:
    print(f"\n Public URL: {url}")
else:
    print("[Warning] URL not ready.")
    print(proc.stderr.read(2000))

[Success] Flask running on port 5000

 Public URL: https://gamma-cash-luck-slots.trycloudflare.com


In [9]:
# API check
import requests, time

time.sleep(2)

try:
    r = requests.get('http://localhost:5000/api/health', timeout=5)
    h = r.json()
    print("[Success] API Health Check")
    print(f"  Status    : {h.get('status')}")
    print(f"  Model     : {h.get('model')}")
    print(f"  K         : {h.get('k')}")
    print(f"  Features  : {h.get('n_features')}")
    print(f"  Reduction : {h.get('reduction')}")
except Exception as e:
    print(f"[Error] Health check failed: {e}")
    print("   Try again later.")

[Success] API Health Check
  Status    : ok
  Model     : KMeans-UMAP
  K         : 6
  Features  : 186
  Reduction : umap
